In [40]:
from scrape_rugbypass import *
import pandas as pd
from bs4 import BeautifulSoup
import requests
import datetime
import os

In [41]:
def derive_custom_url(base_url, text):
    prefix, game_number = base_url.split("?")
    return f"{prefix}{text}/?{game_number}"
    
def derive_stats_url(base_url):
    return derive_custom_url(base_url, "stats")

def derive_teams_url(base_url):
    return derive_custom_url(base_url, "teams")

def derive_dirname(base_url):
    prefix, g = base_url.split("?")
    subdirs = [s for s in prefix.split("/") if s != ""]
    return f"{g[2:]}_{subdirs[-1].replace("-", "_")}"

In [42]:
def transform_teams(teams, side):
    return [{'name': name, **info} for name, info in teams[side].items()]

In [43]:
def merge_stats(team, stats):
    rows = []
    for player in team:
        row = player.copy()
        
        for stat, stat_dict in stats.items():
            row[stat] = stat_dict.get(player['name'], 0)
        rows.append(row)
    return rows

def teams_data(match_id, teams, players):
    home = transform_teams(teams, 'home')
    away = transform_teams(teams, 'away')

    away_performances = pd.DataFrame(merge_stats(away, players))
    home_performances = pd.DataFrame(merge_stats(home, players))

    away_performances["home"] = False
    home_performances["home"] = True

    away_performances["match_id"] = match_id
    home_performances["match_id"] = match_id

    return pd.concat([home_performances, away_performances])

In [44]:
def set_percents(name, data, new):
    for c in ["0", "1", "2", "3"]:
        data[f"{name}_{c}"] = new[c]
    return data
    
def separate_side(match, side):
    data = {}
    special = ["territory_percents", "home_poss_percents", "away_poss_percents"]
    
    for k, v in match.items():
        if k == "territory_percents":
            if side == "home":
                data = set_percents("territory_percents", data, v)
            else:
                data = set_percents("territory_percents", data, 
                    {
                        "0": v["3"],
                        "1": v["2"],
                        "2": v["1"],
                        "3": v["0"]
                    })
        elif k == "home_poss_percents":
            if side == "home":
                data = set_percents("poss_percents", data, v)
        elif k == "away_poss_percents":
            if side == "away":
                data = set_percents("poss_percents", data, v)
        else: 
            data[k] = v[side]

    data["home"] = (side == "home") 
    return data
    
def match_data(match_id, competition_id, base, match):
    base["match_id"] = match_id
    base["competition_id"] = competition_id

    side_stats = pd.DataFrame([separate_side(match, "home"), separate_side(match, "away")])
    side_stats["match_id"] = match_id
    
    return base, side_stats

In [45]:
def scrape_all(match_id, competition_id, base_url):
    stats_url = derive_stats_url(base_url)
    teams_url = derive_teams_url(base_url)

    session = requests.Session()

    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Origin": "https://www.rugbypass.com",
        "Referer": "https://www.rugbypass.com/",
    })
    
    # scrape base info
    response = session.get(base_url)

    dirname = derive_dirname(base_url)
    os.makedirs(f"rugbypass/{dirname}", exist_ok=True)
    
    with open(f"rugbypass/{dirname}/base.html", "w") as f:
        f.write(response.text)
     
    base_data = process_base(response, competition_id)

    # scrape match stats
    response = session.get(stats_url)
    with open(f"rugbypass/{dirname}/stats.html", "w") as f:
        f.write(response.text)
     
    match_stats = process_match_stats(response)
    
    # scrape player stats
    player_stats = scrape_stats(session, stats_url)

    # scrape teams
    teams = scrape_teams(session, teams_url)

    base, side_stats = match_data(match_id, competition_id, base_data, match_stats)
    performances = teams_data(match_id, teams, player_stats)

    return base, side_stats, performances

In [46]:
from tqdm import tqdm

def scrape_urls(urls):
    base_rows = []
    side_rows = []
    perf_rows = []

    for match_id, row in tqdm(urls.iterrows()):
        base, side_stats, perfs = scrape_all(match_id, row["competition_id"], row["url"])

        base_rows.append(base)
        side_rows.append(side_stats)
        perf_rows.append(perfs)

    return base_rows, side_rows, perf_rows

In [47]:
urls = pd.read_parquet("new_urls.parquet")
matches = pd.read_parquet("matches.parquet")
perfs = pd.read_parquet("player_performances.parquet")
missing_match_ids = matches.index[
    ~matches.index.isin(perfs["match_id"])
]

In [63]:
have_team_sheets = urls[~urls.index.isin(missing_match_ids)]

In [73]:
def player_id_from_url(url):
    return url.strip("/").split("/")[-1]

def count_goals(player, name):
    x = player.select_one(f".{name}")
    if x is None:
        return 0
    return len(x.find_all("div", recursive=False))
    
def process_team2(team):
    players = team.select("div.player")

    data = {}
    
    for player in players:
        name = player.select_one(".name").get_text(strip=True)
        number = int(player.select_one(".num").text.strip())

        conversions = count_goals(player, "conversion")
        penalties = count_goals(player, "penalty")
        drop_goals = count_goals(player, "drop_goal")

        player_identifier = player_id_from_url(player.select_one(".name").a["href"])
        
        data[name] = {
            "number": number, 
            "conversions": conversions, 
            "penalty_goals": penalties,
            "drop_goals": drop_goals,
            "player_identifier": player_identifier
        }
        
    return data

def process_teams2(soup):
    teams = soup.select("div.players")

    return {
        "home": process_team2(teams[0]),
        "away": process_team2(teams[1])
    } 

In [65]:
def scrape2(match_id, url):
    teams_url = derive_teams_url(url)

    session = requests.Session()

    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Origin": "https://www.rugbypass.com",
        "Referer": "https://www.rugbypass.com/",
    })

    r = session.get(teams_url)
    html = BeautifulSoup(r.text)

    teams = process_teams2(html)

    home = transform_teams(teams, 'home')
    away = transform_teams(teams, 'away')

    away_performances = pd.DataFrame(away)
    home_performances = pd.DataFrame(home)
    
    away_performances["home"] = False
    home_performances["home"] = True

    away_performances["match_id"] = match_id
    home_performances["match_id"] = match_id

    return pd.concat([home_performances, away_performances])

In [67]:
def scrape_urls2(urls):
    perf_rows = []

    for match_id, row in tqdm(urls.iterrows()):
        perfs = scrape2(match_id, row["url"])
        perf_rows.append(perfs)

    return pd.concat(perf_rows)

In [82]:
kicking_data = scrape_urls2(have_team_sheets)
kicking_data.to_parquet("kicking_data.parquet")

2148it [17:08,  2.09it/s]


In [130]:
player_kicks = pd.read_parquet("kicking_data.parquet")

In [127]:
players = pd.read_parquet("players.parquet")

In [129]:
player_perfs = pd.read_parquet("player_performances.parquet")

In [131]:
player_kicks

,name,number,conversions,penalty_goals,drop_goals,player_identifier,home,match_id
0,Andrew Porter,1,0,0,0,andrew-porter,True,921086
1,Ronan Kelleher,2,0,0,0,ronan-kelleher,True,921086
2,Tadhg Furlong,3,0,0,0,tadhg-furlong,True,921086
3,Tadhg Beirne,4,0,0,0,tadgh-beirne,True,921086
4,James Ryan,5,0,0,0,james-ryan,True,921086
...,...,...,...,...,...,...,...,...
18,Ross Molony,19,0,0,0,ross-molony,False,943609
19,Ted Hill,20,0,0,0,ted-hill,False,943609
20,Louis Schreuder,21,0,0,0,louis-schreuder,False,943609
21,Orlando Bailey,22,0,0,0,orlando-bailey,False,943609


In [143]:
key_cols = ["match_id", "name"]

ambiguous_keys = (
    player_kicks
    .groupby(key_cols)
    .size()
    .loc[lambda x: x > 1]
    .index
)

In [144]:
ambiguous_keys

MultiIndex([(938490, 'Tevita Tatafu'),
            (938559, 'Tevita Tatafu')],
           names=['match_id', 'name'])

In [145]:
kicks_lookup = (
    player_kicks[
        ~player_kicks.set_index(key_cols).index.isin(ambiguous_keys)
    ][["match_id", "name", "player_identifier"]]
)

In [146]:
kicks_lookup

,match_id,name,player_identifier
0,921086,Andrew Porter,andrew-porter
1,921086,Ronan Kelleher,ronan-kelleher
2,921086,Tadhg Furlong,tadhg-furlong
3,921086,Tadhg Beirne,tadgh-beirne
4,921086,James Ryan,james-ryan
...,...,...,...
18,943609,Ross Molony,ross-molony
19,943609,Ted Hill,ted-hill
20,943609,Louis Schreuder,louis-schreuder
21,943609,Orlando Bailey,orlando-bailey


In [193]:
perfs = player_perfs.reset_index(drop=True)

perfs = perfs.merge(
    kicks_lookup,
    on=["match_id", "name"],
    how="left",
    validate="many_to_one",
)

In [195]:
collision_resolve = {
    69835: "tevita-tatafu",
    69853: "tevita-tatafu-1",
    73004: "tevita-tatafu-1",
    73032: "tevita-tatafu"
}

In [196]:
perfs["player_identifier"] = (
    perfs["player_identifier"]
    .fillna(perfs.index.to_series().map(collision_resolve))
)

In [202]:
merged = perfs.merge(
    player_kicks.drop(columns=["name", "number", "home"]),
    on=["match_id", "player_identifier"],
    how="inner",
    validate="one_to_one",
    suffixes=("", "_kick"),
)

In [205]:
players

,player_identifier,name
player_id,,
0,andrew-porter,Andrew Porter
1,ronan-kelleher,Ronan Kelleher
2,tadhg-furlong,Tadhg Furlong
3,tadgh-beirne,Tadhg Beirne
4,james-ryan,James Ryan
...,...,...
3479,tyler-offiah,Tyler Offiah
3480,charlie-mccaig,Charlie McCaig
3481,arthur-relton,Arthur Relton


In [212]:
# create a series indexed by the textual player_id with values the numerical player_ids 
player_id_map = players["player_identifier"].reset_index().set_index(
    "player_identifier"
)["player_id"]

In [213]:
player_id_map

player_identifier
andrew-porter          0
ronan-kelleher         1
tadhg-furlong          2
tadgh-beirne           3
james-ryan             4
                    ... 
tyler-offiah        3479
charlie-mccaig      3480
arthur-relton       3481
ben-douglas         3482
gareth-blackmore    3483
Name: player_id, Length: 3484, dtype: int64

In [214]:
merged["player_id"] = merged["player_identifier"].map(player_id_map)

In [218]:
merged.index.name = "perf_id"
merged = (
    merged
    .drop(columns=["name", "player_identifier"])
)

In [221]:
list(merged.columns)

['number',
 'minutes_played',
 'carries',
 'clean_breaks',
 'completed_tackles',
 'turnovers_conceded',
 'turnovers_won',
 'total_kicks',
 'passes',
 'carries_metres',
 'offloads',
 'defenders_beaten',
 'try_assists',
 'tries',
 'carries_per_minute',
 'total_tackles',
 'missed_tackles',
 'dominant_tackles',
 'ruck_turnovers',
 'lineouts_won',
 'tackles_per_minute',
 'red_cards',
 'yellow_cards',
 'penalties_conceded',
 'home',
 'match_id',
 'conversions',
 'penalty_goals',
 'drop_goals',
 'player_id']

In [226]:
column_order = [
 'match_id', 
 'player_id', 
 'number', 
 'home', 
 'minutes_played',
 'carries',
 'clean_breaks',
 'completed_tackles',
 'turnovers_conceded',
 'turnovers_won',
 'total_kicks',
 'passes',
 'carries_metres',
 'offloads',
 'defenders_beaten',
 'try_assists',
 'tries',
 'carries_per_minute',
 'total_tackles',
 'missed_tackles',
 'dominant_tackles',
 'ruck_turnovers',
 'lineouts_won',
 'tackles_per_minute',
 'red_cards',
 'yellow_cards',
 'penalties_conceded',
 'conversions',
 'penalty_goals',
 'drop_goals'
]

merged = merged[column_order]

In [231]:
merged["number"] = merged["number"].astype(int)

In [252]:
merged.columns

Index(['match_id', 'player_id', 'number', 'home', 'minutes_played', 'carries',
       'clean_breaks', 'completed_tackles', 'turnovers_conceded',
       'turnovers_won', 'total_kicks', 'passes', 'carried_meters', 'offloads',
       'defenders_beaten', 'try_assists', 'tries', 'carries_per_minute',
       'total_tackles', 'missed_tackles', 'dominant_tackles', 'ruck_turnovers',
       'lineouts_won', 'tackles_per_minute', 'red_cards', 'yellow_cards',
       'penalties_conceded', 'conversions', 'penalty_goals', 'drop_goals'],
      dtype='str')

In [256]:
integer_columns = [
    "minutes_played",
    "carries",
    "clean_breaks",
    "completed_tackles",
    "turnovers_conceded",
    "turnovers_won",
    "total_kicks",
    "passes",
    "carried_meters",
    "offloads", 
    "total_tackles",
    "missed_tackles",
    "dominant_tackles",
    "ruck_turnovers",
    "lineouts_won",
    "red_cards",
    "yellow_cards",
    "penalties_conceded",
    "conversions",
    "penalty_goals",
    "drop_goals"
]

for col in integer_columns:
    merged[col] = merged[col].astype(int)